In [1]:
#import sys
#!{sys.executable} -m pip install plotly      #falls man noch was installieren muss, geht das hiermit recht leicht

In [2]:
import requests
import urllib.parse as up
import pandas
from pandas import json_normalize
import json
base = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs"
query = {
    "angebotsart": "1",
    "wo": "Schleswig-Holstein",
    #"umkreis": "200",
    #"arbeitszeit": "ho;mj",  # Semikolon bleibt bewusst erhalten
    #"page": "5",
    "size": "250",
    #"pav": "false",
}
# safe=';' sorgt dafür, dass das Semikolon nicht encodiert wird, falls die API es so erwartet
url = f"{base}?{up.urlencode(query, safe=';')}"

headers = {
    "X-API-Key": "jobboerse-jobsuche",
    #"Accept": "application/json",
    #"User-Agent": "Mozilla/5.0",
}

r = requests.get(url, headers=headers, timeout=30)
print("URL:", r.url)
print("Status:", r.status_code, r.reason)
print("CT:", r.headers.get("content-type"))
data= r.json()


URL: https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs?angebotsart=1&wo=Schleswig-Holstein&size=250
Status: 200 OK
CT: application/json;charset=UTF-8


In [3]:
print(data.keys())

dict_keys(['stellenangebote', 'maxErgebnisse', 'page', 'size', 'woOutput', 'facetten'])


In [4]:
facetten=data["facetten"] #alle möglichen Eigenschaften inklusive der dort vorhanden Daten denke ich mal für Angebotsart 1
#facetten                  #aus manchen Eigenschaften werde ich aber noch nicht so richtig schlau, da keine Einheiten gegeben sind

In [5]:
#print(data.get("stellenangebote"))

In [6]:
jobs=data["stellenangebote"]
#jobs

In [7]:
df_job_SH = json_normalize(jobs)
#df_job_SH

In [8]:
df_job_SH = df_job_SH[["beruf",	"titel",	"refnr",	"arbeitgeber",	"aktuelleVeroeffentlichungsdatum",	"modifikationsTimestamp",	"eintrittsdatum",	"arbeitsort.plz",	"arbeitsort.ort",	"arbeitsort.strasse",	"arbeitsort.region",	"arbeitsort.land",	"arbeitsort.koordinaten.lat",	"arbeitsort.koordinaten.lon",	"arbeitsort.ortsteil",	"externeUrl"]]
#df_job_SH

KeyError: "['arbeitsort.ortsteil'] not in index"

In [ ]:
all_jobs = []

for page in range(1, 3):  # immer Seite 1 als Minimum haben - geht aber auch Seite 41 bis 81
    query["page"] = page   #oben habe ich size 250 gesetzt, daher kriegt man 250 x Seitenanzahl an Ergebnissen
    url = f"{base}?{up.urlencode(query, safe=';')}"  #250 x 40 = 10000
    
    r = requests.get(url, headers=headers, timeout=30)
    data = r.json()
    
    all_jobs.extend(data["stellenangebote"])

df_job_SH = json_normalize(all_jobs) #erzeugt die Spaltenform die man unten sehen kann
df_job_SH = df_job_SH[["beruf",	#alle Spalten die man in der Rückgabe haben will - habe zB die Hashs der spezifischen Personen 
         "titel",	#mal rausgenommen, da wir damit denke eh nichts machen werden
         #"refnr",	
         #"arbeitgeber",	
         #"aktuelleVeroeffentlichungsdatum",	
         #"modifikationsTimestamp",	
         #"eintrittsdatum",	
         #"arbeitsort.plz",	
         #"arbeitsort.ort",	
         #"arbeitsort.strasse",	
         #"arbeitsort.region",	
         #"arbeitsort.land",	
         #"arbeitsort.koordinaten.lat",	
         #"arbeitsort.koordinaten.lon",	
         #"arbeitsort.ortsteil",	
         #"externeUrl"
        ]]
df_job_SH

In [9]:
df_job_SH.to_csv("jobs_SH_500.csv", index=False)

In [16]:
base = "https://rest.arbeitsagentur.de/jobboerse/bewerbersuche-service/pc/v1/bewerber"
query = {
    #"angebotsart": "",
    #"was": "",
    #"ausbildungsart": "",
    "wo": "Schleswig-Holstein",  
    #"umkreis": "",
    #"angebotsart": "",
    #"arbeitszeit": "",
    #"berufserfahrung": "",
    #"vertragsart": "",
    #"page": "",  #Seiten beginnen bei 0
    "size": "250",
    
}
# safe=';' sorgt dafür, dass das Semikolon nicht encodiert wird, falls die API es so erwartet
url = f"{base}?{up.urlencode(query, safe=';')}"

headers = {
    "X-API-Key": "jobboerse-bewerbersuche-ui",
    #"Accept": "application/json",
    #"User-Agent": "Mozilla/5.0",
}

r = requests.get(url, headers=headers, timeout=30)
print("URL:", r.url)
print("Status:", r.status_code, r.reason)
print("CT:", r.headers.get("content-type"))
data= r.json()


URL: https://rest.arbeitsagentur.de/jobboerse/bewerbersuche-service/pc/v1/bewerber?wo=Schleswig-Holstein&size=250
Status: 200 OK
CT: application/json


In [11]:
print(data.keys())

dict_keys(['bewerber', 'maxErgebnisse', 'page', 'size', 'woOutput', 'facetten'])


In [12]:
bewerber=data["bewerber"]
bewerber

[{'refnr': '10000-1205658640-B',
  'verfuegbarkeitVon': '2026-03-01',
  'aktualisierungsdatum': '2026-03-01T11:38:55.881',
  'veroeffentlichungsdatum': '2026-03-01',
  'stellenart': 'ARBEIT',
  'arbeitszeitModelle': ['VOLLZEIT', 'TEILZEIT'],
  'berufe': ['Helfer/in - Lagerwirtschaft, Transport',
   'Helfer/in - Fahrzeugbau und -instandhaltung'],
  'erfahrung': {'gesamterfahrung': 'P10M'},
  'letzteTaetigkeit': {'bezeichnung': 'Servicekraft - Gastronomie und Gastgewerbe',
   'aktuell': True},
  'hatEmail': False,
  'hatTelefon': False,
  'hatAdresse': False,
  'lokation': {'ort': 'Schleswig',
   'plz': '24837',
   'umkreis': '15',
   'region': 'Schleswig-Holstein',
   'land': 'Deutschland'},
  'mehrereArbeitsorte': False},
 {'refnr': '10000-1205476807-B',
  'verfuegbarkeitVon': '2026-03-01',
  'aktualisierungsdatum': '2026-02-27T12:01:55.948',
  'veroeffentlichungsdatum': '2026-03-01',
  'stellenart': 'ARBEIT',
  'arbeitszeitModelle': ['VOLLZEIT'],
  'berufe': ['Managementassistent/in',

In [13]:
df_bew_SH = json_normalize(bewerber)
df_bew_SH

,refnr,verfuegbarkeitVon,aktualisierungsdatum,veroeffentlichungsdatum,stellenart,arbeitszeitModelle,berufe,hatEmail,hatTelefon,hatAdresse,...,lokation.ort,lokation.plz,lokation.umkreis,lokation.region,lokation.land,expertenKenntnisse,erfahrung.berufsfeldErfahrung,letzteTaetigkeit.jahr,ausbildungen,freierTitelStellengesuch
0,10000-1205658640-B,2026-03-01,2026-03-01T11:38:55.881,2026-03-01,ARBEIT,"[VOLLZEIT, TEILZEIT]","[Helfer/in - Lagerwirtschaft, Transport, Helfe...",False,False,False,...,Schleswig,24837,15,Schleswig-Holstein,Deutschland,NaN,NaN,NaN,NaN,NaN
1,10000-1205476807-B,2026-03-01,2026-02-27T12:01:55.948,2026-03-01,ARBEIT,[VOLLZEIT],"[Managementassistent/in, Assistent/in - Hotelm...",False,False,False,...,"Schenefeld, Bz Hamburg",22869,25,Schleswig-Holstein,Deutschland,"[Textverarbeitung Word (MS Office), Korrespond...","[{'berufsfeld': 'Büro und Sekretariat', 'erfah...",2026,NaN,NaN
2,10000-1205377303-B,2026-03-01,2026-02-23T10:02:38.707,2026-03-01,ARBEIT,[TEILZEIT],"[Onlinemarketing-Manager/in, Werbekaufmann/-frau]",False,False,False,...,"Schellhorn bei Preetz, Holstein",24211,25,Schleswig-Holstein,Deutschland,NaN,"[{'berufsfeld': 'Werbung und Marketing', 'erfa...",2026,"[{'jahr': 2008, 'art': 'Werbekaufmann/-frau'},...",NaN
3,10000-1205376122-B,2026-03-01,2026-02-24T13:56:10.937,2026-03-01,ARBEIT,"[VOLLZEIT, HEIM_TELEARBEIT]","[Controller/in, Produktmanager/in, Betriebswir...",False,False,False,...,Kiel,24116,50,Schleswig-Holstein,Deutschland,NaN,"[{'berufsfeld': 'Rechnungswesen, Controlling u...",2026,"[{'jahr': 2018, 'art': 'Bachelor of Arts - Bet...",NaN
4,10000-1205372394-B,2026-02-04,2026-02-04T09:22:47.436,2026-03-01,ARBEIT,[TEILZEIT],[Schulbegleiter/in],False,False,False,...,Elmshorn,25336,20,Schleswig-Holstein,Deutschland,NaN,NaN,NaN,"[{'jahr': 2019, 'art': 'Fachlehrer/in - allgem...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,10000-1205642322-B,2026-04-01,2026-02-26T15:34:13.08,2026-02-26,ARBEIT,[VOLLZEIT],"[Bautischler/in, Hausmeister/in]",False,True,False,...,Hamweddel,24816,50,Schleswig-Holstein,Deutschland,NaN,"[{'berufsfeld': 'Aus- und Trockenbau, Isolieru...",2026,"[{'jahr': 1989, 'art': 'Tischler/in - Bau- und...",NaN
246,10000-1205642273-B,2026-02-26,2026-02-26T15:24:21.325,2026-02-26,ARBEIT,"[VOLLZEIT, TEILZEIT]","[Helfer/in - Lagerwirtschaft, Transport]",False,False,False,...,Trittau,22946,30,Schleswig-Holstein,Deutschland,NaN,NaN,NaN,NaN,NaN
247,10000-1205641270-B,2026-02-26,2026-02-26T16:29:41.815,2026-02-26,ARBEIT,[VOLLZEIT],[Fahrradmonteur/in],False,False,True,...,Stockelsdorf,23617,20,Schleswig-Holstein,Deutschland,NaN,"[{'berufsfeld': 'Fahrzeug-, Luft-, Raumfahrt- ...",2026,"[{'jahr': 2022, 'art': 'Fahrradmonteur/in'}]",NaN
248,10000-1205641067-B,2026-02-26,2026-02-26T14:41:03.382,2026-02-26,ARBEIT,[VOLLZEIT],"[Maler/in und Lackierer/in - Maler/in, Maler/i...",False,False,False,...,Wentorf bei Sandesneben,23898,50,Schleswig-Holstein,Deutschland,"[Kundenberatung, -betreuung, Arbeitsvorbereitung]","[{'berufsfeld': 'Maler, Stuckateure, Bautensch...",2026,"[{'jahr': 2008, 'art': 'Raumausstatter/in'}]",NaN


In [18]:
all_bewerber = []

for page in range(1, 3):  # immer Seite 1 als Minimum haben - geht aber auch Seite 41 bis 81
    query["page"] = page   #oben habe ich size 250 gesetzt, daher kriegt man 250 x Seitenanzahl an Ergebnissen
    url = f"{base}?{up.urlencode(query, safe=';')}"  #250 x 40 = 10000
    
    r = requests.get(url, headers=headers, timeout=30)
    data = r.json()
    
    all_bewerber.extend(data["bewerber"])

df_bew_SH = json_normalize(all_bewerber) #erzeugt die Spaltenform die man unten sehen kann
df_bew_SH = df_bew_SH[[
    #"refnr",
    #"verfuegbarkeitVon",
    #"aktualisierungsdatum",
    "veroeffentlichungsdatum",
    "stellenart",
    #"arbeitszeitModelle",
    "berufe",
    #"hatEmail",
    #"hatTelefon",
    #"hatAdresse",
    #"mehrereArbeitsorte",
    #"erfahrung.gesamterfahrung",
    "letzteTaetigkeit.jahr",
    "letzteTaetigkeit.bezeichnung",
    #"letzteTaetigkeit.aktuell",
    #"lokation.ort",
    #"lokation.plz",
    #"lokation.umkreis",
    #"lokation.region",
    #"lokation.land",
    #"ausbildungen",
    #"erfahrung.berufsfeldErfahrung",
    #"expertenKenntnisse",
    #"freierTitelStellengesuch"
    ]]
df_bew_SH

,veroeffentlichungsdatum,stellenart,berufe,letzteTaetigkeit.jahr,letzteTaetigkeit.bezeichnung
0,2026-03-01,ARBEIT,"[Helfer/in - Lagerwirtschaft, Transport, Helfe...",NaN,Servicekraft - Gastronomie und Gastgewerbe
1,2026-03-01,ARBEIT,"[Managementassistent/in, Assistent/in - Hotelm...",2026,Managementassistent/in
2,2026-03-01,ARBEIT,"[Onlinemarketing-Manager/in, Werbekaufmann/-frau]",2026,Onlinemarketing-Manager/in
3,2026-03-01,ARBEIT,"[Controller/in, Produktmanager/in, Betriebswir...",2026,Controller/in
4,2026-03-01,ARBEIT,[Schulbegleiter/in],NaN,NaN
...,...,...,...,...,...
495,2026-02-26,ARBEIT,"[Helfer/in - Lagerwirtschaft, Transport, Helfe...",2026,Helfer/in - Reinigung
496,2026-02-26,ARBEIT,"[Helfer/in - Reinigung, Helfer/in - Gartenbau,...",2020,Helfer/in - Reinigung
497,2026-02-26,ARBEIT,"[Helfer/in - Büro, Verwaltung]",2017,Gärtner/in - Garten- und Landschaftsbau
498,2026-02-26,ARBEIT,[Helfer/in - Verkauf],2026,Helfer/in - Verkauf


In [15]:
#df_bew_SH.to_csv("bewerber_SH_500.csv", index=False)                

In [16]:
base = "https://rest.arbeitsagentur.de/infosysbub/absuche/pc/v1/ausbildungsangebot"
query = {
    #"sty": "",
    #"ids": "",
    #"orte": "",
    #"page": "",  
    "size": "250",
    #"uk": "",
    "re": "SLH",
    #"bt": "",
    #"bart": "",
    #"ityp": "",  
    #"ban": "",
    #"bg": ""
    
}
# safe=';' sorgt dafür, dass das Semikolon nicht encodiert wird, falls die API es so erwartet
url = f"{base}?{up.urlencode(query, safe=';')}"

headers = {
    "X-API-Key": "infosysbub-absuche",
    #"Accept": "application/json",
    #"User-Agent": "Mozilla/5.0",
}

r = requests.get(url, headers=headers, timeout=30)
print("URL:", r.url)
print("Status:", r.status_code, r.reason)
print("CT:", r.headers.get("content-type"))
data= r.json()

URL: https://rest.arbeitsagentur.de/infosysbub/absuche/pc/v1/ausbildungsangebot?size=250&re=SLH
Status: 200 OK
CT: application/hal+json


In [17]:
print(data.keys())

dict_keys(['_embedded', '_links', 'page'])


In [18]:
ausbildung=data["_embedded"]
ausbildung

{'termine': [{'id': 369359028,
   'unterrichtsform': {'id': 105,
    'bezeichnung': 'Sonstige Präsenzveranstaltung'},
   'dauer': {'id': 6, 'bezeichnung': 'mehr als 6 Monate bis 1 Jahr'},
   'anbieterbewertung': None,
   'angebot': {'id': 16367897,
    'titel': 'Vorbereitungskurs auf eine Umschulung zum Fachinformatiker inkl. CompTIA A+ Zertifizierung und Stressmanagement',
    'inhalt': '<p>Die IT-Branche wächst rasant und verlangt nach zertifizierten Fachkräften mit belastbarer Kompetenz. Der Vorbereitungskurs auf die Umschulung zum Fachinformatiker vermittelt fundiertes Wissen, integriert die CompTIA A+ Zertifizierung und stärkt mit Skills im Bereich Stressmanagement die persönliche Widerstandskraft.<br/></p><h4>Basics: Selbstorganisation und Teamwork am Arbeitsplatz</h4><p> (Dauer: ca. 4 Wochen)<br/></p><ul><li> Arbeits-/Selbstorganisation</li><li> Lern-/Arbeitstechniken</li><li> Kommunikation/Konflikte im Team</li><li> Zeit-/Stressmanagement</li></ul><h4>First & Second-Level-Suppo

In [19]:
df_aus_SH = json_normalize(ausbildung)
df_aus_SH

,termine
0,"[{'id': 369359028, 'unterrichtsform': {'id': 1..."
